In [1]:
# هدف: شناسایی فایل‌های پردازش‌شده و آماده‌سازی مسیرهای پروژه برای تحلیل بازتولیدپذیر

from pathlib import Path
import pandas as pd

project_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "processed").exists()
)

data_processed = project_root / "data" / "processed"
outputs_figures = project_root / "outputs" / "figures"
outputs_tables = project_root / "outputs" / "tables"

outputs_figures.mkdir(parents=True, exist_ok=True)
outputs_tables.mkdir(parents=True, exist_ok=True)

expected_files = [
    "isfahan_final_cooling_priority_500m.gpkg",
    "lst_isfahan_summer_median_2023_2025.tif",
    "ndvi_isfahan_summer_median_2023_2025.tif",
    "ghs_built_surface_isfahan_2020.tif",
    "ghs_population_isfahan_2020.tif",
]

inventory = pd.DataFrame({
    "file_name": expected_files,
    "exists": [(data_processed / name).exists() for name in expected_files],
    "path": [str(data_processed / name) for name in expected_files],
})

display(inventory)

,file_name,exists,path
0,isfahan_final_cooling_priority_500m.gpkg,True,F:\Isfahan_Urban_Heat_Cooling_Prioritization\d...
1,lst_isfahan_summer_median_2023_2025.tif,True,F:\Isfahan_Urban_Heat_Cooling_Prioritization\d...
2,ndvi_isfahan_summer_median_2023_2025.tif,True,F:\Isfahan_Urban_Heat_Cooling_Prioritization\d...
3,ghs_built_surface_isfahan_2020.tif,True,F:\Isfahan_Urban_Heat_Cooling_Prioritization\d...
4,ghs_population_isfahan_2020.tif,True,F:\Isfahan_Urban_Heat_Cooling_Prioritization\d...


In [3]:
# هدف: کنترل مشخصات مکانی، ابعاد و دامنهٔ مقادیر رسترها و شبکهٔ نهایی اولویت‌بندی

import numpy as np
import rasterio
import geopandas as gpd

raster_files = [
    "lst_isfahan_summer_median_2023_2025.tif",
    "ndvi_isfahan_summer_median_2023_2025.tif",
    "ghs_built_surface_isfahan_2020.tif",
    "ghs_population_isfahan_2020.tif",
]

raster_audit = []

for file_name in raster_files:
    file_path = data_processed / file_name

    with rasterio.open(file_path) as src:
        values = src.read(1, masked=True)
        valid_values = values.compressed()

        raster_audit.append({
            "file_name": file_name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution": src.res,
            "min": float(np.min(valid_values)),
            "mean": float(np.mean(valid_values)),
            "max": float(np.max(valid_values)),
            "nodata": src.nodata,
        })

raster_audit_df = pd.DataFrame(raster_audit)
display(raster_audit_df)

final_grid_path = data_processed / "isfahan_final_cooling_priority_500m.gpkg"
final_grid = gpd.read_file(final_grid_path)

print("Final grid CRS:", final_grid.crs)
print("Number of 500 m cells:", len(final_grid))
print("Number of attributes:", len(final_grid.columns))
display(final_grid.head())

,file_name,crs,width,height,resolution,min,mean,max,nodata
0,lst_isfahan_summer_median_2023_2025.tif,EPSG:32639,1064,1166,"(30.0, 30.0)",36.657585,50.505512,62.292736,-9.999000e+03
1,ndvi_isfahan_summer_median_2023_2025.tif,EPSG:32639,1064,1166,"(30.0, 30.0)",-0.097638,0.137146,0.746528,-9.999000e+03
2,ghs_built_surface_isfahan_2020.tif,ESRI:54009,37,38,"(1000.0, 1000.0)",0.000000,141294.466546,493563.000000,4.294967e+09
3,ghs_population_isfahan_2020.tif,ESRI:54009,37,38,"(1000.0, 1000.0)",0.000000,2921.234387,13711.062480,-2.000000e+02


Final grid CRS: EPSG:32639
Number of 500 m cells: 2296
Number of attributes: 17


,coverage_ratio,grid_id,grid_area_km2,isfahan_grid_ghsl_features_population_3x3_sum,isfahan_grid_ghsl_features_population_3x3_mean,isfahan_grid_ghsl_features_population_valid_cells,isfahan_grid_ghsl_features_built_surface_3x3_sum,isfahan_grid_ghsl_features_built_surface_3x3_mean,isfahan_grid_ghsl_features_built_surface_valid_cells,isfahan_grid_ghsl_features_built_surface_fraction_3x3,hot_mean,lowveg_mean,cooling_need,population_exposure,built_exposure,final_priority_score,geometry
0,0.253999,ISF_0001,0.063500,3979.896265,442.210696,9,500918.0,55657.555556,9,0.055658,0.0,0.378378,0.151351,0,0,0.105946,"MULTIPOLYGON (((549500 3625000, 549500 3624500..."
1,0.195923,ISF_0002,0.048981,3979.896265,442.210696,9,500918.0,55657.555556,9,0.055658,0.0,0.049180,0.019672,0,0,0.013770,"MULTIPOLYGON (((549500 3625500, 549500 3625000..."
2,0.287913,ISF_0003,0.071978,5168.811518,574.312391,9,663245.0,73693.888889,9,0.073694,0.0,0.966292,0.386517,0,0,0.270562,"MULTIPOLYGON (((549500 3626000, 549500 3625500..."
3,0.416477,ISF_0004,0.104119,5168.811518,574.312391,9,663245.0,73693.888889,9,0.073694,0.0,0.864407,0.345763,0,0,0.242034,"MULTIPOLYGON (((549500 3626000, 549307.574 362..."
4,0.508496,ISF_0005,0.127124,2310.482744,256.720305,9,223808.0,24867.555556,9,0.024868,0.0,0.379562,0.151825,0,0,0.106277,"MULTIPOLYGON (((549500 3624500, 550000 3624500..."


In [4]:
# هدف: تولید جدول آمار توصیفی شاخص‌های مورد استفاده در اولویت‌بندی مداخلات خنک‌کننده

summary_variables = {
    "نسبت پوشش سلول (%)": ("coverage_ratio", 100),
    "مساحت سلول (کیلومتر مربع)": ("grid_area_km2", 1),
    "میانگین تراکم جمعیت (نفر در کیلومتر مربع)": (
        "isfahan_grid_ghsl_features_population_3x3_mean", 1
    ),
    "سهم سطح ساخته‌شده (%)": (
        "isfahan_grid_ghsl_features_built_surface_fraction_3x3", 100
    ),
    "سهم نواحی داغ حرارتی (%)": ("hot_mean", 100),
    "سهم نواحی کم‌پوشش‌گیاهی (%)": ("lowveg_mean", 100),
    "امتیاز نیاز به مداخلهٔ خنک‌کننده": ("cooling_need", 1),
    "امتیاز نهایی اولویت": ("final_priority_score", 1),
}

summary_rows = []

for indicator, (field, multiplier) in summary_variables.items():
    values = final_grid[field] * multiplier

    summary_rows.append({
        "شاخص": indicator,
        "تعداد سلول‌های معتبر": int(values.notna().sum()),
        "کمینه": values.min(),
        "میانگین": values.mean(),
        "میانه": values.median(),
        "انحراف معیار": values.std(),
        "بیشینه": values.max(),
        "مقادیر گمشده": int(values.isna().sum()),
    })

table_02 = pd.DataFrame(summary_rows)

numeric_columns = [
    "کمینه", "میانگین", "میانه", "انحراف معیار", "بیشینه"
]
table_02[numeric_columns] = table_02[numeric_columns].round(3)

table_02_csv = outputs_tables / "table_02_descriptive_statistics_500m_grid.csv"
table_02_xlsx = outputs_tables / "table_02_descriptive_statistics_500m_grid.xlsx"

table_02.to_csv(table_02_csv, index=False, encoding="utf-8-sig")
table_02.to_excel(table_02_xlsx, index=False)

display(table_02)

print(f"CSV saved to: {table_02_csv}")
print(f"Excel saved to: {table_02_xlsx}")

,شاخص,تعداد سلول‌های معتبر,کمینه,میانگین,میانه,انحراف معیار,بیشینه,مقادیر گمشده
0,نسبت پوشش سلول (%),2296,10.244,95.674,100.000,15.579,100.000,0
1,مساحت سلول (کیلومتر مربع),2296,0.026,0.239,0.250,0.039,0.250,0
2,میانگین تراکم جمعیت (نفر در کیلومتر مربع),2296,0.000,2848.961,1505.586,3095.671,10837.032,0
3,سهم سطح ساخته‌شده (%),2296,0.000,13.868,9.978,12.247,43.532,0
4,سهم نواحی داغ حرارتی (%),2296,0.000,20.589,0.000,33.598,100.000,0
5,سهم نواحی کم‌پوشش‌گیاهی (%),2296,0.000,20.275,6.629,28.498,100.000,0
6,امتیاز نیاز به مداخلهٔ خنک‌کننده,2296,0.000,0.205,0.073,0.271,1.000,0
7,امتیاز نهایی اولویت,2296,0.000,0.253,0.300,0.180,0.729,0


CSV saved to: F:\Isfahan_Urban_Heat_Cooling_Prioritization\outputs\tables\table_02_descriptive_statistics_500m_grid.csv
Excel saved to: F:\Isfahan_Urban_Heat_Cooling_Prioritization\outputs\tables\table_02_descriptive_statistics_500m_grid.xlsx
